# 03. Run a tournament

Walks through the full tournament toolchain:

1. **Write** a JSON config (1 RL agent + 2 scripted bots, 1 iteration).
2. **Run** the tournament -> CSV.
3. **Parse** the CSV into a structured JSON.
4. **Visualize**: generate the PDF tree (basic + game-theoretic metrics).
5. **Inspect** the produced artefacts.

End to end: ~30-60 s on CPU. For the full 19-agent thesis tournament,
use `microrts-agent tournament run single_map` (uses
[`microrts_agent/tournament_configs/single_map.json`](../microrts_agent/tournament_configs/single_map.json),
takes hours).

Prereq: [`00_navigate.ipynb`](00_navigate.ipynb) must be green.

## 1. Write the config

Tournament configs are JSON. The full shipped config has 19 AIs over 5
iterations = 1710 games. For the notebook we subset to 3 AIs over 1
iteration.

**Schema** (every field has a clear role):

- `maps`: list of map XML paths.
- `ais`: list of AI references. RL agents are `agent:<path-to-dir>`
  (path relative to the config file or absolute). Scripted bots are
  bare names from `microrts_agent/registries/ai.py` (`AI_MAPPING`).
- `iterations`: how many times the full round-robin is repeated.
- `maxGameLengths`: list aligned with `maps`, in env steps.
- `timeBudget` (ms): per-decision wallclock budget for scripted bots.

In [ ]:
import json

from microrts_agent.paths import PROJECT_ROOT

agent_dir = PROJECT_ROOT / "data" / "agents" / "UECD-SingleMap-Best"
out_dir = PROJECT_ROOT / "outputs" / "tournaments" / "notebook-mini"
out_dir.mkdir(parents=True, exist_ok=True)
config_path = out_dir / "notebook-mini.json"

config = {
    "maps": ["maps/open_competition/basesWorkers16x16A.xml"],
    "ais": [
        f"agent:{agent_dir}",  # UECD-SingleMap-Best
        "WorkerRush",  # weak scripted bot
        "CoacAI",  # strong scripted bot
    ],
    "iterations": 1,
    "maxGameLengths": [4000],
    "timeBudget": 100,
    "iterationsBudget": -1,
    "preAnalysisBudget": 3_600_000,
    "fullObservability": True,
    "selfMatches": False,
    "timeoutCheck": False,
    "runGC": False,
    "saveTraces": False,
    "saveGameLogs": False,
    "slowAIs": [],
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Config written to:", config_path.relative_to(PROJECT_ROOT))
print("AIs:", config["ais"])
print("Games to play: 3 AIs x 2 others / 2 (deduped) x 2 positions x 1 iter = 6")

## 2. Run the tournament

`microrts-agent tournament run <config-path-or-name>` reads the JSON
above, plays the games, writes `tournament.csv` next to the config.

In [ ]:
import subprocess

result = subprocess.run(
    ["microrts-agent", "tournament", "run", str(config_path)],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=600,
)
print(result.stdout)
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr)

## 3. Parse the CSV

The runner's CSV is human-readable but awkward to query. `tournament
parse` converts it to a structured `tournament_parsed.json` (one record
per game), which is what all the visualisation tooling consumes.

In [ ]:
result = subprocess.run(
    ["microrts-agent", "tournament", "parse", str(out_dir)],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=60,
)
print(result.stdout[-800:])

parsed_path = out_dir / "tournament_parsed.json"
with open(parsed_path) as f:
    parsed = json.load(f)
print(f"\nGames recorded: {len(parsed['games'])}\n")
for g in parsed["games"]:
    p = g["players"]
    r = g["result"]
    winner = (
        p["ai1"]["name"] if r["winner"] == 0 else p["ai2"]["name"] if r["winner"] == 1 else "draw"
    )
    print(f"  {p['ai1']['name']:25s} vs {p['ai2']['name']:25s} -> {winner} ({r['time']} steps)")

## 4. Generate the PDF visualisations

`tournament viz` produces the same PDF tree as the shipped
[`data/tournaments/single_map/visualizations/`](../data/tournaments/single_map/visualizations/),
but on our mini tournament: standings, head-to-head matrix, game-theoretic
metrics (Nash equilibrium, regret, exploitability).

`tournament analyze` is the shorthand for `parse` + `viz` in one call.

In [ ]:
result = subprocess.run(
    ["microrts-agent", "tournament", "viz", str(out_dir)],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=120,
)
print(result.stdout)

## 5. Inspect the produced artefacts

In [ ]:
for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        rel = path.relative_to(out_dir)
        size_kb = path.stat().st_size / 1024
        print(f"  {size_kb:>10.1f} KB  {rel}")

## Next steps

- Full thesis-scale tournament:
  `microrts-agent tournament run single_map` (uses
  [`microrts_agent/tournament_configs/single_map.json`](../microrts_agent/tournament_configs/single_map.json),
  19 AIs x 5 iter = 1710 games, runs for hours).
- Browse the pre-rendered thesis output under
  [`data/tournaments/single_map/visualizations/`](../data/tournaments/single_map/visualizations/).
- For multi-map (5 maps x 16 AIs x 5 iter = 6000 games):
  [`microrts_agent/tournament_configs/multi_map.json`](../microrts_agent/tournament_configs/multi_map.json).